# Preliminary Results

**Project title:** *No Breathing Room: Spatial Modeling of Air Pollution Exposure Along Jeepney Routes in Quezon City*

**Course and section: CS 138 WFY-2**

**Consultation date: November 20, 2026, 6:00 PM - 6:15 PM**

## 1. Member Contributions and Signatures

| Member | Contributions | Signature |
| --- | --- | --- |
|ALBANO, Elliot Xander | | |
|CELESTINO, Leo | Drafted Section 3.2 writeup and potential no-neighbor case for IDW | |
|PACTOLIN, Renz Sergio | Added setup for getting weather data | |
|PADILLA, Klex Felouise|Proposal Summary, Setup for OpenAQ, Correspondence with CCESD, Draft for Section 4.3| |
|SALTING, Ambernie Green | Added setup for spatiotemporal IDW | |

## 2. Proposal Summary

### 2.1 Problem and Significance

This project investigates how commuting conditions relate to estimated exposure to fine particulate matter (PM2.5). Exposure during a trip depends on the pollutant concentrations encountered and the time spent along different road segments. Thus, a shorter or faster trip may not necessarily have the lowest estimated exposure. Research on commuting exposure provides the motivation for examining these differences (Panchal et al., 2022).

The study focuses on selected jeepney routes in Quezon City. It proposes combining available air-quality observations, environmental information, and route characteristics to estimate pollution along these corridors and simulate how concentrations change under simplified wind and emission conditions. The intended contribution is a computational proof of concept for comparing route exposure and examining the assumptions behind those estimates. Its outputs will be interpreted as model-based estimates, rather than direct measurements of passengers' inhaled dose or a validated air-quality forecasting service.

### 2.2 Research Questions and Hypotheses

The study retains the two research questions stated in the revised proposal:

1. How do route characteristics and environmental conditions influence estimated commuter exposure to air pollution, and which factors are most strongly associated with the estimated exposure?
2. How do pollutant concentrations evolve spatially and temporally under different wind and pollutant-source scenarios, and how can the resulting concentration estimates be used to compare commuting routes?

The following working hypotheses make the expected relationships explicit:

- **H1:** For otherwise comparable trips, routes that involve more time in areas with higher PM2.5 concentrations will have greater estimated cumulative exposure. Differences in concentration along a route may affect comparisons based on travel time or distance alone.
- **H2:** Changing wind conditions or pollutant-source intensity will change the simulated spatial and temporal distribution of PM2.5, potentially changing the relative exposure estimates for selected routes or route segments.

### 2.3 Scope, Study Area, and Jeepney Routes

The pollutant of interest is PM2.5, and the proposed study area covers selected commuting corridors in Quezon City. The revised proposal identifies three jeepney routes as areas of interest using Sakay.ph (By Implication Inc., n.d.):

- Katipunan to the UP Diliman campus, and vice versa;
- SM Fairview to Philcoa, and vice versa;
- SM North to the UP Diliman campus, and vice versa.

These are candidate study corridors. The final geographic extent, observation period, time resolution, and routes retained for analysis will depend on usable data coverage. Because the three routes serve different endpoints, comparisons across them will describe exposure under specified trip conditions.

The proposal identifies Quezon City air-quality resources and OpenAQ as possible sources of numerical PM2.5 observations, ERA5 as a possible source of weather information, and MMDA traffic-volume records as inputs to the pollutant-source model.

If monitoring coverage is insufficient, the study will narrow its domain or use explicitly labelled simplified simulations. If traffic data or a justified conversion from traffic volume to emissions is unavailable, pollutant sources will represent hypothetical scenarios. Interpretation will remain limited by monitoring coverage, route and travel-time information, and the assumptions of the two-dimensional model.

## 3. Stuff We Tried and Want to Try

### 3.1 Data Acquisition and Preparation

In [ ]:
%pip install requests pandas

In [ ]:
from __future__ import annotations
from pathlib import Path
import pandas as pd
from requests import get
from pandas import DataFrame
from dataclasses import dataclass


layout:
- email correspondence with CCESD
- FOI requested

#### OpenAQ: June 2026 daily PM2.5

Load the saved CSV in `OpenAQ-06-2026`; no API key or OpenAQ request is needed when running this notebook. The file already contains daily averages from location **4526988**, Near Katipunan Ave., Quezon City. Do not average these rows again. To refresh the download, run `retrieve_june.py` from inside that directory, then rerun the loading cell below.

Use the **start** of each reporting period in Asia/Manila as its calendar date: the interval June 30 to July 1 belongs to June 30. Keep `hours_observed` and `coverage_percent` alongside each mean so incomplete days remain identifiable.

In [ ]:
# Support kernels started in the repository, its parent, or the data directory.
project_candidates = [
    Path.cwd(),
    Path.cwd() / 'cs138-capstone-wfy2',
    Path.cwd().parent,
]
csv_relative = Path('OpenAQ-06-2026/openaq_4526988_june_2026_daily_pm25.csv')
project_dir = next(
    (root for root in project_candidates if (root / csv_relative).is_file()),
    None,
)
if project_dir is None:
    raise FileNotFoundError(f'Cannot find {csv_relative} from {Path.cwd()}')

pm25_raw = pd.read_csv(project_dir / csv_relative)
pm25_daily = pm25_raw.rename(columns={'pm25': 'pm25_mean'}).copy()
for column in ['datetime_from', 'datetime_to']:
    pm25_daily[column] = (
        pd.to_datetime(pm25_daily[column], utc=True).dt.tz_convert('Asia/Manila')
    )
pm25_daily['date'] = pm25_daily['datetime_from'].dt.strftime('%Y-%m-%d')
pm25_daily['location_id'] = 4526988
pm25_daily = pm25_daily.loc[
    pm25_daily['date'].between('2026-06-01', '2026-06-30')
].sort_values(['date', 'sensor_id']).reset_index(drop=True)
if pm25_daily.duplicated(['sensor_id', 'date']).any():
    raise ValueError('Expected one daily average per sensor and date.')

print(f'Loaded {len(pm25_daily)} daily PM2.5 observations.')
print(pm25_daily[[
    'date', 'sensor_id', 'pm25_mean', 'units',
    'hours_observed', 'coverage_percent',
]].head().to_string(index=False))

### 3.2 Spatiotemporal IDW and k-d Tree Search

#### Implemented concentration estimations using Spatiotemporal IDW

`spatiotemporal_idw.py` implements a version of the IDW algorithm from Li et al. (2014). Obtaining the _k_ nearest neighbors using a 3D version of the k-d tree below, it calculates a PM2.5 concentration at a location (x,y,t) using the following formula: 
\begin{equation}
w(x, y, t) = \sum_{i=1}^{N} \lambda_iw_i
\end{equation}
where $\lambda_i$ is the weight assigned to each monitoring station, and \[w_i\] is the actual PM2.5 concentration recorded at the monitoring station from our data. 

The station weight is assigned to each station by calculating: 
\begin{equation}
\lambda_i = \frac{(\frac{1}{d_i})^p}{\sum_{i=1}^N (\frac{1}{d_i})^p}
\end{equation}
where $d_i$ is the distance metric: 

\begin{equation}
d_i = \sqrt{(x_i - x)^2 + (y_i - y)^2 + c^2(t_i - t)^2}
\end{equation}
- $x, y, t$ are the coordinates of each neighbor station 
- $c$ is the conversion factor with from time units to spatial distance units. 

#### Implemented neighbor search

`knearestneighbors.py` implements a **static, two-dimensional k-d tree**. `buildTree` sorts each active slice along its splitting axis, stores the median point and its original integer ID, and recursively builds the two children. Axes alternate X/Y. Cai, Xu, and Zhang (2021, [Section III-B, Algorithm 2](https://arxiv.org/html/2102.10808v1#S3.SS2)) also describe median-based construction, but choose a maximal-covariance axis and maintain incremental-update metadata. Our implementation uses alternating axes and has no insertion, deletion, or incremental rebalancing. Its repeated sorting gives an O(n log² n) build cost; balanced depth does not guarantee a logarithmic search in every case.

Search visits the nearer subtree first and keeps at most `k` candidates in a heap. It visits the farther subtree when the heap is not full or the squared distance to the splitting plane is no greater than the worst candidate's squared distance. This is the bounded-neighbor search principle described by Li et al. (2014, [Sections 2.4.5–2.4.6](https://www.mdpi.com/1660-4601/11/9/9101)). Searching may still visit every point in the worst case.

Use `nearest_neighbors(root, query, k, exclude_id=None)` for station work: each result retains `id`, `point`, and `distance_sq`, sorted by `(distance_sq, id)`. Equal-distance ties keep the lower ID. Pass an original station ID only when excluding that station, such as a future leave-one-station-out check; a different station at the same coordinates remains eligible. `k=0` and empty trees return no neighbors; oversized `k` returns all eligible points; negative `k` is invalid. `KNNOne` and `KNNAll` retain their coordinate-only interfaces. `KNNAll` now excludes the original input ID and returns results in input order despite tree reordering.

The Python mechanics use read-only `@property` accessors, `Enum` axis labels, and a lambda sorting key ([Python property documentation](https://docs.python.org/3.13/library/functions.html#property), [Enum documentation](https://docs.python.org/3/library/enum.html), [lambda expressions](https://docs.python.org/3.13/tutorial/controlflow.html#lambda-expressions)).

In [ ]:
# Run Section 3.1 first so project_dir and the saved OpenAQ data are available.
import sys

if sys.version_info < (3, 10):
    raise RuntimeError('Use Python 3.10 or newer for this notebook.')
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from knearestneighbors import Axis, KNNAll, KNNOne, buildTree, nearest_neighbors

#### Select one date before constructing a spatial tree

This implementation accepts only `(x, y)`, with **no time coordinate**. First select observations whose reporting-period start falls on the requested Asia/Manila date, as prepared in Section 3.1. Keep the measurement's `location_id`, `sensor_id`, PM2.5 mean, units, and coverage together. Require one eligible PM2.5 value per physical location for that date; multiple sensors at one site need an explicit quality/aggregation policy rather than becoming separate spatial stations. Apply any justified coverage rule before building a tree, and rebuild for each date's available stations. Pooling all 30 days would repeat one spatial location, not supply 30 stations. We note that this is a sequence of spatial snapshots, not the continuous space–time extension used by Li et al. (2014).

For real distances, obtain verified station coordinates and their source CRS/datum, then transform both stations and route/grid query points into the **same suitable projected CRS with meter units** before calling `buildTree`. Store projected `x_m, y_m` with the original location ID; retrieve concentrations by that ID, never by reordered list position. A suitable local UTM projection is one option ([PROJ UTM documentation](https://proj.org/en/stable/operations/projections/utm.html)); latitude/longitude degrees must not be passed to this Euclidean distance calculation as meters. Record the actual CRS chosen with future results.

The current CSV has no coordinates and contains one location. The following cell demonstrates date selection only; it does not build a measured spatial tree. OpenAQ is currently our **only PM2.5 measurement source**, as CCESD has not provided any measurements yet.

In [ ]:
interpolation_date = '2026-06-15'
daily_station_observations = pm25_daily.loc[
    pm25_daily['date'].eq(interpolation_date)
].copy()
if daily_station_observations.empty:
    raise ValueError(f'No saved PM2.5 observations for {interpolation_date}.')
if daily_station_observations.duplicated('location_id').any():
    raise ValueError('Resolve multiple PM2.5 sensors per location before spatial selection.')

print(f'{interpolation_date}: {daily_station_observations.location_id.nunique()} monitoring location(s).')
print(daily_station_observations[[
    'date', 'location_id', 'sensor_id', 'pm25_mean', 'units',
    'hours_observed', 'coverage_percent',
]].to_string(index=False))
print('The saved CSV has no station coordinates; no measured spatial tree is built here.')

#### Synthetic demonstration — not measured pollution results

All five IDs, coordinates, and example concentrations below are **invented test inputs** on an artificial Cartesian grid in meters. They are separate from the OpenAQ table and do not represent Quezon City monitoring locations. The small query shows how retained IDs select the corresponding example PM2.5 values after tree construction changes point order. It performs neighbor selection only.

In [ ]:
synthetic_stations = pd.DataFrame([
    # station_id, x_m, y_m, pm25_example (fictional, in micrograms per cubic meter)
    (90, 3000.0, 2000.0, 14.0),
    (10,    0.0,    0.0, 10.0),
    (70, 2000.0,    0.0, 25.0),
    (30,    0.0, 2000.0, 20.0),
    (50, 1000.0, 1000.0, 18.0),
], columns=['station_id', 'x_m', 'y_m', 'pm25_example'])
if not synthetic_stations['station_id'].is_unique:
    raise ValueError('Synthetic station IDs must be unique.')
synthetic_by_id = synthetic_stations.set_index('station_id')
synthetic_points = [
    (int(row.station_id), (row.x_m, row.y_m))
    for row in synthetic_stations.itertuples(index=False)
]
synthetic_tree_points = synthetic_points.copy()  # buildTree reorders this copy.
synthetic_root = buildTree(synthetic_tree_points, 0, len(synthetic_tree_points), Axis.X)
synthetic_query_m = (900.0, 800.0)
synthetic_neighbors = nearest_neighbors(synthetic_root, synthetic_query_m, k=3)
synthetic_selection = pd.DataFrame([
    {
        'station_id': neighbor.id,
        'distance_m': neighbor.distance_sq ** 0.5,
        'pm25_example': synthetic_by_id.loc[neighbor.id, 'pm25_example'],
    }
    for neighbor in synthetic_neighbors
])
print('SYNTHETIC neighbor selection — fictional values, not measured PM2.5 or IDW output')
print(synthetic_selection.to_string(index=False))

#### How the selection will feed IDW

For a future date-specific IDW estimate, look up each selected station's same-date concentration by ID and use its distance $d_i=\sqrt{\mathrm{distance\_sq}_i}$ in meters. For nonzero distances and a chosen exponent $p>0$, the planned weighted mean is

$$\widehat C(q,t)=\frac{\sum_{i\in N_k(q,t)}d_i^{-p}C_i(t)}{\sum_{i\in N_k(q,t)}d_i^{-p}}.$$

Li et al. (2014, [Section 2.2.1](https://www.mdpi.com/1660-4601/11/9/9101)) give inverse-distance weights and combine space and time; the expression here is the same-date spatial specialization. IDW still needs weighting code, an exact-location rule to avoid division by zero, a quality policy for conflicting co-located values, and justified choices of `k`, `p`, and search extent. A unique valid zero-distance station would supply its observed value directly. Missing eligible stations should yield an explicitly unavailable estimate.

This integration implements and checks **neighbor selection**, not completed IDW interpolation. Multiple distinct monitoring locations with overlapping dates, verified coordinates/CRS, consistent units, coverage/quality information, and projected route/grid query points are needed before spatial interpolation is meaningful. Enough stations must remain after holding out locations to evaluate prediction accuracy. No IDW pollution map or measured interpolation-error score is reported from the present single-location dataset.

### 3.3 Numerical Linear Algebra Formulation and Methods

### 3.4 Advection-Diffusion PDE and Finite Difference Scheme

The proposed PDE model will simulate PM2.5 transport across a simplified horizontal domain containing the selected route corridors. It represents wind-driven transport, effective diffusion, and pollutant introduction through

$$
\frac{\partial C}{\partial t}+u\frac{\partial C}{\partial x}+v\frac{\partial C}{\partial y}
=D\left(\frac{\partial^2 C}{\partial x^2}+\frac{\partial^2 C}{\partial y^2}\right)+S(x,y,t),
$$

where:

- $C(x,y,t)$ is the PM2.5 concentration at horizontal position $(x,y)$ and time $t$.
- $u$ and $v$ are the wind-velocity components in the $x$- and $y$-directions, respectively.
- $D$ is the nonnegative effective diffusion coefficient, which controls the spreading of pollutants.
- $S(x,y,t)$ is the pollutant-source term, expressed as concentration added per unit time.
- $\frac{\partial C}{\partial t}$ represents the change in concentration over time at a fixed location.
- $u\frac{\partial C}{\partial x}+v\frac{\partial C}{\partial y}$ represents advection, or pollutant transport by wind.
- $D\left(\frac{\partial^2 C}{\partial x^2}+\frac{\partial^2 C}{\partial y^2}\right)$ represents diffusion, or the spreading of pollutants from higher toward lower concentrations.

Initially, $D$ will be treated as constant within each scenario. Traffic volume alone is not an emission rate; its conversion into $S$ requires additional assumptions and parameters. Hypothetical sources will be labeled accordingly.

**Domain and inputs.** A rectangular grid in projected meter coordinates will use spacings $\Delta x,\Delta y$ and time step $\Delta t$. The notation $C^n_{i,j}$ denotes concentration at grid point $(i,j)$ and time $n\Delta t$. Wind speed and direction from the weather workflow will require conversion into components consistent with the grid axes, units, and direction convention. The final domain, resolution, and coefficients remain to be selected.

**Initial and boundary conditions.** An IDW field may initialize the model once sufficient spatial observations are available. **The current daily OpenAQ series comes from one monitoring location and cannot establish a measured spatial initial field.** Boundary conditions must be selected and justified before simulation; their effects on transport into and out of the domain will be documented.

**Finite difference procedure.** Following the proposal's explicit finite difference approach, with Mahadi and Puzi (2022) as a methodological reference, time derivatives will use forward Euler stepping, advection will use upwind differences selected by the signs of $u,v$, and diffusion will use central differences. Each step will compute interior concentrations from the previous time level, add the source contribution, and enforce the chosen boundary conditions.

**Upwind differences.** An upwind difference estimates a spatial derivative using the neighboring point on the side the wind comes from, matching the direction of pollutant transport ([*Numerical Methods for Engineers*, Section 8.3](https://leifh.folk.ntnu.no/teaching/tkt4140/._main070.html)). For wind blowing from left to right across points A, B, and C, the advection calculation at B uses A and B. In our notation:

- If $u>0$ (wind toward increasing $x$), use the left neighbor, giving a backward spatial difference:

  $$\frac{\partial C}{\partial x}\approx\frac{C^n_{i,j}-C^n_{i-1,j}}{\Delta x}.$$

- If $u<0$ (wind toward decreasing $x$), use the right neighbor, giving a forward spatial difference:

  $$\frac{\partial C}{\partial x}\approx\frac{C^n_{i+1,j}-C^n_{i,j}}{\Delta x}.$$

The same rule applies in the $y$-direction using the sign of $v$. If a velocity component is zero, its advection contribution is zero. Here, backward and forward refer to spatial neighbors. The diffusion terms retain central differences.

**Verification and outputs.** Before route analysis, the combined scheme's stability restriction will be derived to select compatible grid and time steps. Controlled cases and spatial/time refinement will check transport behavior, nonnegative concentrations, and convergence. Wind/source scenarios will produce concentration maps and snapshots for Section 3.5's exposure calculation.

### 3.5 Route Exposure Estimation and Comparison

### 3.6 Unsuccessful Attempts, Obstacles, and Planned Experiments

## 4. Results with Data Commentary

### 4.1 Computational Setup and Reproducibility

#### Python compatibility and dependencies

The notebook's generic `Python 3` kernelspec does not pin a Python version. The integrated module and notebook code target **Python 3.10+**, validation for this integration uses the available **Python 3.13.1** interpreter. The runtime-report cell below records the versions in the active kernel. The module uses a regular `Point = tuple[float, float]` alias rather than the Python 3.12+ `type` statement.

Actual third-party notebook dependencies are **pandas** for the saved CSV/weather tables and **requests** for the existing optional weather download. The `%pip install requests pandas` cell in Section 3.1 installs these in the active Jupyter/IPython kernel. `knearestneighbors.py` is a local module, with `heapq`, `enum`, `dataclasses`, `pathlib`, being standard Python library modules.

Run Section 3.1 setup/imports and OpenAQ loading, then all Section 3.2 cells, this runtime report, and the Section 4.2-4.3 checks. These steps use the saved CSV and synthetic test inputs without API downloads. The existing Section 4.6 weather retrieval is separate and requires network access if deliberately run; it is not required for the k-d tree checks. A Jupyter installation is needed to run the `.ipynb` interactively.

In [ ]:
import platform
from importlib.metadata import version

print(f'Python {platform.python_version()}')
for package_name in ['pandas', 'requests']:
    print(f'{package_name} {version(package_name)}')
print('knearestneighbors.py: local module; algorithm dependencies: Python standard library')

### 4.2 Data Availability, Quality, and Preprocessing

In [ ]:
# Summarize coverage per sensor without filling missing observations with zero.
pm25_coverage = pm25_daily.groupby('sensor_id').agg(
    days_present=('date', 'nunique'),
    days_with_pm25=('pm25_mean', 'count'),
    min_hours_observed=('hours_observed', 'min'),
    max_hours_observed=('hours_observed', 'max'),
    min_coverage_percent=('coverage_percent', 'min'),
    max_coverage_percent=('coverage_percent', 'max'),
)
pm25_coverage['missing_days'] = 30 - pm25_coverage['days_present']
print(pm25_coverage.to_string())

The saved June dataset contains 30 daily observations. All dates are present, but no daily mean has a complete 24 hours of observations. The loading step preserves all daily means and their coverage; no missing hours are replaced with zero and no coverage cutoff is imposed.

All 30 rows are temporal observations from **one monitoring location** (location 4526988) close to Katipunan Avenue, not 30 spatial stations. OpenAQ is currently our only PM2.5 measurement source; **CCESD has not provided any PM2.5 measurements yet**. Spatial interpolation across the study corridors requires multiple distinct monitoring locations observed on matching dates, verified coordinates and CRS, consistent units, and coverage/quality metadata. The current CSV supplies no coordinates and cannot support a measured multi-station interpolation or spatial validation.

### 4.3 IDW Pollution Maps and Interpolation Evaluation

#### Neighbor-selection correctness checks - synthetic inputs only

Compare returned station IDs, coordinates, and distances with an independent brute-force calculation using `math.hypot`, sorted by distance and original ID. These are correctness checks of the implemented search, **not PM2.5 interpolation validation**. Cases include: 
- reordered input, 
- self-exclusion, 
- co-located distinct IDs, 
- equal distances across branches, 
- empty/singleton and collinear trees, 
- `k=0`, 
- oversized/invalid `k`, 
- seeded random queries. 

The `KNNAll` regression checks original input identity after tree construction reorders its working list. The demo's ID-to-value association is also checked.

In [ ]:
from math import hypot, isclose
from random import Random


def check_against_brute_force(samples, query, k, exclude_id=None):
    # Independent baseline: no reuse of the module's distance or heap code.
    expected = sorted(
        (hypot(point[0] - query[0], point[1] - query[1]), station_id, point)
        for station_id, point in samples if station_id != exclude_id
    )[:k]
    working_points = samples.copy()
    root = buildTree(working_points, 0, len(working_points), Axis.X)
    actual = nearest_neighbors(root, query, k, exclude_id=exclude_id)
    assert [item.id for item in actual] == [item[1] for item in expected]
    assert [item.point for item in actual] == [item[2] for item in expected]
    for item, (expected_distance, _, _) in zip(actual, expected):
        assert isclose(item.distance_sq ** 0.5, expected_distance,
                       rel_tol=1e-12, abs_tol=1e-9)
    assert KNNOne(root, query, k, exclude_id) == [item[2] for item in expected]


synthetic_edge_sets = [
    [],
    [(91, (0.0, 0.0))],
    # Deliberately unsorted IDs; duplicate coordinates and equal-distance ties.
    [(42, (1.0, 0.0)), (0, (0.0, 0.0)), (7, (-1.0, 0.0)),
     (3, (0.0, 1.0)), (99, (0.0, -1.0)), (5, (0.0, 0.0))],
    [(80 - i, (0.0, float(i))) for i in range(8)],
]
neighbor_checks = 0
for samples in synthetic_edge_sets:
    for query in [(0.0, 0.0), (0.5, -0.5), (250.0, -80.0)]:
        for excluded in [None] + [station_id for station_id, _ in samples]:
            for k in [0, 1, 2, len(samples) + 2]:
                check_against_brute_force(samples, query, k, excluded)
                neighbor_checks += 1

rng = Random(138)
random_ids = rng.sample(range(1000, 9000), 64)
synthetic_random_points = [
    (station_id, (rng.uniform(-5000, 5000), rng.uniform(-5000, 5000)))
    for station_id in random_ids
]
for _ in range(24):
    query = (rng.uniform(-6000, 6000), rng.uniform(-6000, 6000))
    for k in [0, 1, 5, 67]:
        check_against_brute_force(synthetic_random_points, query, k)
        neighbor_checks += 1
for station_id, query in synthetic_random_points:
    for k in [1, 5, 67]:
        check_against_brute_force(synthetic_random_points, query, k, station_id)
        neighbor_checks += 1

# Regression for the original i[0] failure and exclusion after point reordering.
original_points = [point for _, point in synthetic_edge_sets[2]]
original_points_copy = original_points.copy()
for k in [0, 1, 3, 10]:
    all_results = KNNAll(original_points, k)
    assert [query for query, _ in all_results] == original_points_copy
    for original_id, (query, neighbors) in enumerate(all_results):
        expected = sorted(
            (hypot(point[0] - query[0], point[1] - query[1]), other_id, point)
            for other_id, point in enumerate(original_points_copy)
            if other_id != original_id
        )[:k]
        assert neighbors == [item[2] for item in expected]
assert original_points == original_points_copy
assert KNNAll([], 3) == []
assert KNNAll([(0.0, 0.0)], 3) == [((0.0, 0.0), [])]

for bad_k, expected_error in [(-1, ValueError), (1.5, TypeError)]:
    for operation in [
        lambda: nearest_neighbors(None, (0.0, 0.0), bad_k),
        lambda: KNNOne(None, (0.0, 0.0), bad_k),
        lambda: KNNAll([], bad_k),
    ]:
        try:
            operation()
        except expected_error:
            pass
        else:
            raise AssertionError(f'Expected {expected_error.__name__} for k={bad_k}')

# Tree nodes and the selected example values must refer to the same original ID.
assert synthetic_tree_points != synthetic_points
assert [item.id for item in synthetic_neighbors] == [50, 10, 70]
assert synthetic_selection['pm25_example'].tolist() == [18.0, 10.0, 25.0]
for item in synthetic_neighbors:
    station = synthetic_by_id.loc[item.id]
    assert item.point == (station.x_m, station.y_m)

print(f'PASS: {neighbor_checks} SYNTHETIC queries match brute-force IDs and distances.')
print('PASS: KNNAll original-ID regression, edge/invalid k cases, and ID-to-value association.')
print('These checks establish search correctness only; no measured IDW evaluation was performed.')

Passing the assertions establishes agreement with brute-force neighbor selection on these finite synthetic cases. It does not establish a speed advantage, pollution prediction accuracy, or real monitoring coverage. No performance claim or interpolated pollution map is made. A later IDW implementation should be evaluated on held-out monitoring **locations** with overlapping dates after additional measurements and coordinate metadata become available; holding out days from the single current site would not test spatial interpolation.

### 4.4 Numerical Linear Algebra Results and Diagnostics

### 4.5 PDE Simulation Setup: Parameters, Initial Conditions, and Boundary Conditions

### 4.6 PDE Results: Wind and Source Scenarios

In [ ]:

@dataclass(frozen=True)
class WindData:
    wind_speed: float
    wind_direction: float

url = "https://archive-api.open-meteo.com/v1/archive"

# QC Circle
params = {
    "latitude": 14.6507,
    "longitude": 121.0494,
    "start_date": "2026-01-01",
    "end_date": "2026-06-30",
    "daily": [
        "wind_speed_10m_mean",
        "wind_direction_10m_dominant"
    ],
    "hourly": [
        "wind_speed_10m",
        "wind_direction_10m"
    ],
    "wind_speed_unit": "ms",
    "timezone": "Asia/Manila"
}

response = get(url, params=params)
response.raise_for_status()
data = response.json()

##
## DAILY DATA (to csv)
df_daily = DataFrame(data["daily"])
df_daily = df_daily.rename(columns={
    "time": "date",
    "wind_speed_10m_mean": "wind_speed",
    "wind_direction_10m_dominant": "wind_direction"
})
df_daily.to_csv("qc_memorial_circle_daily_wind.csv", index=False)

##
## HOURLY DATA (to csv)
df_hourly = DataFrame(data["hourly"])
df_hourly = df_hourly.rename(columns={
    "time": "timestamp",
    "wind_speed_10m": "wind_speed",
    "wind_direction_10m": "wind_direction"
})
df_hourly.to_csv("qc_memorial_circle_hourly_wind.csv", index=False)

##
##  DICTS
daily_weather: dict[str, WindData] = {
    row.date: WindData(row.wind_speed, row.wind_direction)
    for row in df_daily.itertuples(index=False)
}

hourly_weather: dict[tuple[str, str], WindData] = {
    tuple(row.timestamp.split("T")): WindData(
        row.wind_speed,
        row.wind_direction
    )
    for row in df_hourly.itertuples(index=False)
}


## DEBUG: Example Accesses
print(daily_weather["2026-01-01"])
print(hourly_weather[("2026-01-01", "00:00")])


#### Align daily PM2.5 with daily wind

Run the weather cell above after the OpenAQ loading cell in Section 3.1. Both datasets use Manila calendar dates. The left join below keeps every PM2.5 observation and selects its corresponding daily wind record from the longer weather period. This aligns dates between the Katipunan sensor and QC Memorial Circle weather series; the observations refer to different locations. The resulting `pm25_weather_daily` table is available for subsequent analysis.

In [ ]:
weather_daily = df_daily[['date', 'wind_speed', 'wind_direction']].copy()
weather_daily['date'] = pd.to_datetime(weather_daily['date']).dt.strftime('%Y-%m-%d')
pm25_weather_daily = pm25_daily.merge(
    weather_daily, on='date', how='left', validate='many_to_one',
)
missing_wind = pm25_weather_daily[['wind_speed', 'wind_direction']].isna().any(axis=1)
print(f'PM2.5 rows without complete daily wind data: {missing_wind.sum()}')
print(pm25_weather_daily[[
    'date', 'pm25_mean', 'units', 'coverage_percent',
    'wind_speed', 'wind_direction',
]].head().to_string(index=False))

### 4.7 Numerical Stability, Accuracy, and Convergence

### 4.8 Jeepney Route Exposure Comparisons

### 4.9 Overall Data Commentary: Successful and Unsuccessful Results, Assumptions, and Limitations

## 5. Questions for the Instructor

### Question 1

### Question 2

## References

The following references are carried over from the revised proposal for Section 2. Li et al. (2014) also supports the neighbor-search and planned IDW discussion in Section 3.2. Additional implementation sources from `references.txt` follow below.

- Alvarez-Vazquez, L., Garcia-Chan, N., Martinez, A., & Vazquez-Mendez, M. (2017). Numerical simulation of air pollution due to traffic flow in urban networks. *Journal of Computational and Applied Mathematics, 326*, 44-61. https://doi.org/10.1016/j.cam.2017.05.017
- By Implication Inc. (n.d.). *Jeepney routes*. Sakay.ph. Retrieved October 29, 2026, from https://explore.sakay.ph/jeeps
- Li, L., Losser, T., Yorke, C., & Piltner, R. (2014). Fast inverse distance weighting-based spatiotemporal interpolation: A web-based application of interpolating daily fine particulate matter PM2.5 in the contiguous U.S. using parallel programming and k-d tree. *International Journal of Environmental Research and Public Health, 11*(9), 9101-9141. https://doi.org/10.3390/ijerph110909101
- Mahadi, N. N. I., & Puzi, S. M. (2022). Numerical solutions of advection-diffusion equation using explicit finite difference method. *Proceedings of Science and Mathematics, 11*, 136-143. https://science.utm.my/procscimath/wp-content/uploads/sites/605/2022/10/136-143-Nurul-Nur-Izzaty_Shazirawati-Mohd-Puzi.pdf
- Panchal, R., Panagi, M., May, H. R., Obszynska, J. A., Evans, M. S., Hansell, A. L., Gulliver, J., & Hey, J. D. V. (2022). Personal air pollution exposure during morning commute car and active transport journeys. *Journal of Transport & Health, 26*, 101365. https://doi.org/10.1016/j.jth.2022.101365
- *Numerical Methods for Engineers*. (n.d.). 8.3 Upwind schemes. Norwegian University of Science and Technology (NTNU). https://leifh.folk.ntnu.no/teaching/tkt4140/._main070.html. Supports the upwind explanation and spatial difference formulas in Section 3.4.

#### k-d tree and Python implementation sources

- Cai, Y., Xu, W., & Zhang, F. (2021). *ikd-Tree: An incremental k-d tree for robotic applications*. arXiv:2102.10808. [Paper](https://arxiv.org/abs/2102.10808), [Section III-B / Algorithm 2](https://arxiv.org/html/2102.10808v1#S3.SS2). Supports the median-construction comparison; its maximal-covariance splitting and incremental features are not implemented here.
- Python Software Foundation. (n.d.). *heapq — Heap queue algorithm*. [Python 3.13 documentation](https://docs.python.org/3.13/library/heapq.html), [Python 3.14 documentation](https://docs.python.org/3.14/library/heapq.html). Supports the min-heap implementation and the documented introduction of public max-heap functions in 3.14.
- Python Software Foundation. (n.d.). *enum — Support for enumerations*. [Documentation](https://docs.python.org/3/library/enum.html). Supports the X/Y axis labels, not k-d tree or IDW theory.
- Python Software Foundation. (n.d.). *Built-in functions: property* and *More control flow tools: lambda expressions*. [Properties](https://docs.python.org/3.13/library/functions.html#property), [lambda expressions](https://docs.python.org/3.13/tutorial/controlflow.html#lambda-expressions). Canonical documentation for the accessors and sorting key.
- PROJ contributors. (n.d.). *Universal Transverse Mercator (UTM)*. [Documentation](https://proj.org/en/stable/operations/projections/utm.html). Background for converting geodetic coordinates to a suitable local projected coordinate system; no current station coordinates are inferred from it.

#### Coding aids references

- GeeksforGeeks. (2025). *Python property decorator — @property*. [Tutorial](https://www.geeksforgeeks.org/python/python-property-decorator-property/). Coding aid for `TreeNode` accessors.
- GeeksforGeeks. (2025). *Getter and setter in Python*. [Tutorial](https://www.geeksforgeeks.org/python/getter-and-setter-in-python/). Related accessor background; this module does not add setters.
- GeeksforGeeks. (2026). *Python lambda functions*. [Tutorial](https://www.geeksforgeeks.org/python/python-lambda-anonymous-functions-filter-map-reduce/). Coding aid for the axis sorting key.
- GeeksforGeeks. (2025). *Search and insertion in K dimensional tree*. [Tutorial](https://www.geeksforgeeks.org/dsa/search-and-insertion-in-k-dimensional-tree/). Introductory axis-partitioning background; not a source for the bounded k-nearest-neighbor or IDW procedures.

## LLM Usage Declaration

Padilla, Klex Felouise:

1. Python Notebook Initial Template
- LLM: Codex
- Purpose: Generating initial Python Notebook template
- Prompt: https://chatgpt.com/s/cx_6af14145dc588191a8a5e2dbafd3df32
- Output: `Preliminary_Results_Template.ipynb`. It has since been edited for brevity in this Python notebook (`cs138-wfy2-preliminaryresults.ipynb`).

2. OpenAQ Data Retrieval Assistance
- LLM: Codex
- Purpose: Retrieving PM2.5 data for June 2026, aggregated into daily averages, and integrating the saved CSV with coverage summaries and daily weather data in this notebook
- Prompt: https://chatgpt.com/s/cx_6afee7b4842481919c5e9a63edee9ec2
- Output: `OpenAQ-06-2026/retrieve_june.py` and OpenAQ loading, coverage, and weather-alignment cells in `cs138-wfy2-preliminaryresults.ipynb`

3. k-d Tree Notebook Integration Assistance
- LLM: Codex
- Purpose: Help with integrating and testing Elliot's k-d tree implementation with the Python Notebook.
- Prompt: https://chatgpt.com/s/cx_6aff0a1fc7e4819180955b7a210f345a
- Output: Edits in `knearestneighbors.py`, `references.txt`, `cs138-wfy2-preliminaryresults.ipynb`

Albano, Elliot Xander:

- LLM: ChatGPT
- Purpose: Debugging k-d tree implementation
1. First debugging wave:  https://chatgpt.com/share/6afdb672-0e90-83e8-846d-920cf188d9af
2. Second debugging wave: https://chatgpt.com/share/6afdb65f-ff1c-83ee-82a1-205de6839e80
3. Final debugging:       https://chatgpt.com/share/6afdb649-4980-83e8-9203-84411ae23b1d
- Output: `knearestneighbors.py`